In [1]:
# CELL 2 ? Imports
import json
import os
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

# Hugging Face
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset, load_dataset

# Evaluation
from sklearn.metrics import classification_report, confusion_matrix

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"GPU available: {torch.cuda.is_available()}")


Using device: cuda
GPU available: True


In [2]:
print("Loading FEVER dataset...")
fever = load_dataset("fever", "v1.0", trust_remote_code=True)

print("\nDataset splits:")
print(fever)

AVAILABLE_SPLITS = list(fever.keys())
print(f"\nAvailable splits: {AVAILABLE_SPLITS}")

print("\nOne example from training set:")
example = fever['train'][0]
for key, val in example.items():
    print(f"  {key}: {val}")


Loading FEVER dataset...

Dataset splits:
DatasetDict({
    train: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 311431
    })
    labelled_dev: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 37566
    })
    unlabelled_dev: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 19998
    })
    unlabelled_test: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 19998
    })
    paper_dev: Dataset({
        features: ['id', 'label', 'claim', 'evidence_annotation_id', 'evidence_id', 'evidence_wiki_url', 'evidence_sentence_id'],
        num_rows: 18999
    })
   

In [3]:
LABEL_MAP = {
    'SUPPORTS':        0,   # → support
    'REFUTES':         1,   # → contradict
    'NOT ENOUGH INFO': 2,   # → neutral
}

ID_TO_LABEL = {
    0: 'support',
    1: 'contradict',
    2: 'neutral',
}

print("Label mapping:")
for fever_label, our_id in LABEL_MAP.items():
    print(f"  FEVER '{fever_label}' → id {our_id} → our label '{ID_TO_LABEL[our_id]}'")

Label mapping:
  FEVER 'SUPPORTS' → id 0 → our label 'support'
  FEVER 'REFUTES' → id 1 → our label 'contradict'
  FEVER 'NOT ENOUGH INFO' → id 2 → our label 'neutral'


In [4]:
# CELL 5 — Look at real examples from each class

train_data = fever['train']

# Count labels
from collections import Counter
label_counts = Counter(train_data['label'])
print("Label distribution in training set:")
for label, count in label_counts.items():
    pct = count / len(train_data) * 100
    print(f"  {label}: {count:,} examples ({pct:.1f}%)")

print(f"\nTotal training examples: {len(train_data):,}")

# Show one example per class
print("\n--- Example SUPPORTS ---")
for ex in train_data:
    if ex['label'] == 'SUPPORTS':
        print(f"  Claim:    {ex['claim']}")
        print(f"  Evidence: {ex.get('evidence_wiki_url', 'N/A')}")
        break

print("\n--- Example REFUTES ---")
for ex in train_data:
    if ex['label'] == 'REFUTES':
        print(f"  Claim:    {ex['claim']}")
        break

print("\n--- Example NOT ENOUGH INFO ---")
for ex in train_data:
    if ex['label'] == 'NOT ENOUGH INFO':
        print(f"  Claim:    {ex['claim']}")
        break

Label distribution in training set:
  SUPPORTS: 193,756 examples (62.2%)
  REFUTES: 70,066 examples (22.5%)
  NOT ENOUGH INFO: 47,609 examples (15.3%)

Total training examples: 311,431

--- Example SUPPORTS ---
  Claim:    Nikolaj Coster-Waldau worked with the Fox Broadcasting Company.
  Evidence: Nikolaj_Coster-Waldau

--- Example REFUTES ---
  Claim:    Adrienne Bailon is an accountant.

--- Example NOT ENOUGH INFO ---
  Claim:    System of a Down briefly disbanded in limbo.


In [5]:
# CELL 6 ? Prepare data for training
#
# Important note:
# The Hugging Face FEVER builder available here exposes Wikipedia page titles
# via `evidence_wiki_url`, not the original evidence sentence text.
# So this notebook trains a stronger weak-supervision baseline:
# - SUPPORTS / REFUTES use linked Wikipedia page titles as weak evidence text
# - NOT ENOUGH INFO uses hard negatives sampled from lexically related but different titles
# - the training subset is balanced across the three labels
# This is still weaker than training on true evidence sentences, but it is much less inflated
# than the original placeholder-based setup.

import random
import re
from collections import defaultdict, Counter

RNG = random.Random(42)
TOKEN_RE = re.compile(r"[a-z0-9]{3,}")


def normalize_wiki_title(value: str) -> str:
    return ' '.join((value or '').replace('_', ' ').split()).strip()


def text_tokens(text: str) -> set[str]:
    return set(TOKEN_RE.findall((text or '').lower()))


def collect_title_pool(split_data, max_scan=None):
    titles = []
    seen = set()
    for i, item in enumerate(split_data):
        if max_scan and i >= max_scan:
            break
        title = normalize_wiki_title(item.get('evidence_wiki_url', ''))
        if title and title.lower() not in seen:
            seen.add(title.lower())
            titles.append(title)
    return titles


def sample_negative_title(claim: str, fallback_titles: list[str]) -> str:
    claim_tokens = text_tokens(claim)
    scored = []
    for title in fallback_titles:
        title_tokens = text_tokens(title)
        if not title_tokens:
            continue
        overlap = len(claim_tokens & title_tokens)
        if overlap == 0:
            continue
        if title.lower() in claim.lower():
            continue
        scored.append((overlap, title))

    if scored:
        scored.sort(key=lambda item: (item[0], len(item[1])), reverse=True)
        top_band = [title for overlap, title in scored[: min(100, len(scored))]]
        return RNG.choice(top_band)

    candidates = [title for title in fallback_titles if title.lower() not in claim.lower()]
    if not candidates:
        candidates = fallback_titles or ['Unrelated Wikipedia topic']
    return RNG.choice(candidates)


def prepare_fever_examples(split_data, fallback_titles, max_examples=None):
    buckets = defaultdict(list)
    target_per_class = max_examples // 3 if max_examples else None

    for item in split_data:
        label_str = item.get('label', '')
        if label_str not in LABEL_MAP:
            continue
        if target_per_class and len(buckets[label_str]) >= target_per_class:
            continue

        claim = ' '.join((item.get('claim', '') or '').split()).strip()
        if not claim:
            continue

        evidence_title = normalize_wiki_title(item.get('evidence_wiki_url', ''))
        if label_str == 'NOT ENOUGH INFO':
            evidence_text = sample_negative_title(claim, fallback_titles)
            evidence_source = 'sampled_hard_negative_title'
        else:
            evidence_text = evidence_title or sample_negative_title(claim, fallback_titles)
            evidence_source = 'wiki_title_weak_supervision' if evidence_title else 'sampled_fallback_title'

        buckets[label_str].append({
            'claim': claim,
            'evidence_text': evidence_text,
            'label': LABEL_MAP[label_str],
            'label_str': ID_TO_LABEL[LABEL_MAP[label_str]],
            'evidence_source': evidence_source,
        })

        if target_per_class and all(len(buckets[label]) >= target_per_class for label in LABEL_MAP):
            break

    examples = []
    for label_str in ['SUPPORTS', 'REFUTES', 'NOT ENOUGH INFO']:
        examples.extend(buckets[label_str])

    RNG.shuffle(examples)
    return examples


def pick_validation_split(dataset_dict):
    for candidate in ['paper_dev', 'validation', 'dev', 'labelled_dev']:
        if candidate in dataset_dict:
            return candidate
    raise RuntimeError(f'No validation/dev split found. Available splits: {list(dataset_dict.keys())}')


if torch.cuda.is_available():
    MAX_TRAIN = 15000
    MAX_VAL = 2000
else:
    MAX_TRAIN = 6000
    MAX_VAL = 1200
    print('CPU detected. Using a smaller balanced subset. For best results, use GPU.')

val_split_name = pick_validation_split(fever)
print(f'Using validation split: {val_split_name}')

print('Building title pool for harder neutral examples...')
title_pool = collect_title_pool(fever['train'], max_scan=150000)
print(f'Title pool size: {len(title_pool):,}')

print(f'Preparing {MAX_TRAIN:,} balanced training examples...')
train_examples = prepare_fever_examples(fever['train'], title_pool, max_examples=MAX_TRAIN)
val_examples = prepare_fever_examples(fever[val_split_name], title_pool, max_examples=MAX_VAL)

print(f'Training examples:   {len(train_examples):,}')
print(f'Validation examples: {len(val_examples):,}')
print('Training label distribution:', Counter(e['label_str'] for e in train_examples))
print('Validation label distribution:', Counter(e['label_str'] for e in val_examples))

if train_examples:
    print('\nPrepared example:')
    ex = train_examples[0]
    for k, v in ex.items():
        print(f'  {k}: {v}')
else:
    raise RuntimeError('No training examples were prepared. Check the FEVER split and preprocessing logic.')


Using validation split: paper_dev
Building title pool for harder neutral examples...
Title pool size: 9,509
Preparing 15,000 balanced training examples...
Training examples:   15,000
Validation examples: 1,998
Training label distribution: Counter({'contradict': 5000, 'neutral': 5000, 'support': 5000})
Validation label distribution: Counter({'neutral': 666, 'support': 666, 'contradict': 666})

Prepared example:
  claim: Haiti declined membership in the World Trade Organization.
  evidence_text: Haiti
  label: 1
  label_str: contradict
  evidence_source: wiki_title_weak_supervision


In [6]:
MODEL_NAME = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Test the tokenizer
test_claim = "The sky is blue"
test_evidence = "Scientists confirm that the sky appears blue due to Rayleigh scattering"

encoded = tokenizer(
    test_claim,
    test_evidence,
    max_length=320,
    truncation=True,
    padding='max_length',
    return_tensors='pt',
)

print(f"\nTokenizer output keys: {list(encoded.keys())}")
print(f"input_ids shape: {encoded['input_ids'].shape}")
print(f"Token count: {encoded['attention_mask'].sum().item()}")
print(f"\nDecoded back: {tokenizer.decode(encoded['input_ids'][0][:30])}")


Loading tokenizer: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli

Tokenizer output keys: ['input_ids', 'token_type_ids', 'attention_mask']
input_ids shape: torch.Size([1, 320])
Token count: 18

Decoded back: [CLS] The sky is blue[SEP] Scientists confirm that the sky appears blue due to Rayleigh scattering[SEP][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD][PAD]


In [7]:
# CELL 8 ? Tokenize all examples
#
# We feed BOTH the claim and evidence together so the model learns their relationship.

MAX_LENGTH = 320  # Slightly longer context for better claim-title matching.


def tokenize_examples(examples):
    if not examples:
        raise RuntimeError('No examples available for tokenization.')

    claims = [e['claim'] for e in examples]
    evidence_texts = [e['evidence_text'] for e in examples]
    labels = [e['label'] for e in examples]

    encodings = tokenizer(
        claims,
        evidence_texts,
        max_length=MAX_LENGTH,
        truncation=True,
        padding='max_length',
        return_tensors='pt',
    )

    dataset_dict = {
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': torch.tensor(labels, dtype=torch.long),
    }

    if 'token_type_ids' in encodings:
        dataset_dict['token_type_ids'] = encodings['token_type_ids']

    return Dataset.from_dict({k: v.numpy() for k, v in dataset_dict.items()})


print('Tokenizing training set...')
train_dataset = tokenize_examples(train_examples)

print('Tokenizing validation set...')
val_dataset = tokenize_examples(val_examples)

print(f"\nTrain dataset: {train_dataset}")
print(f"Val dataset:   {val_dataset}")

if len(train_dataset) > 0:
    print(f"\nOne training record keys: {train_dataset[0].keys()}")
    print(f"Label of first record: {train_dataset[0]['labels']} -> {ID_TO_LABEL[train_dataset[0]['labels']]}")
else:
    raise RuntimeError('Training dataset is empty after tokenization.')


Tokenizing training set...
Tokenizing validation set...

Train dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'token_type_ids'],
    num_rows: 15000
})
Val dataset:   Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'token_type_ids'],
    num_rows: 1998
})

One training record keys: dict_keys(['input_ids', 'attention_mask', 'labels', 'token_type_ids'])
Label of first record: 1 -> contradict


In [8]:
# CELL 9 — Load the model
#
# AutoModelForSequenceClassification automatically adds a 
# classification head on top of the transformer:
#
#   [BERT/DeBERTa transformer]
#         ↓
#   [pooling: take [CLS] token]
#         ↓
#   [linear layer: hidden_size → 3]
#         ↓
#   [3 output logits: support, contradict, neutral]

print(f"Loading model: {MODEL_NAME}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,          # support, contradict, neutral
    ignore_mismatched_sizes=True,  # allows reusing pretrained weights with new head
)

model = model.to(device)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model on device:      {device}")

Loading model: MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli

Total parameters:     184,424,451
Trainable parameters: 184,424,451
Model on device:      cuda


In [9]:
# CELL 10 ? Evaluation metrics
#
# Track metrics that are more informative than raw accuracy alone.

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    from sklearn.metrics import f1_score, precision_recall_fscore_support, balanced_accuracy_score

    accuracy = (predictions == labels).mean()
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    precision, recall, _, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)

    return {
        'accuracy': round(float(accuracy), 4),
        'f1_macro': round(float(f1_macro), 4),
        'balanced_accuracy': round(float(balanced_acc), 4),
        'precision_macro': round(float(precision), 4),
        'recall_macro': round(float(recall), 4),
    }


dummy_logits = np.array([[2.0, 0.1, 0.3], [0.1, 1.8, 0.2]])
dummy_labels = np.array([0, 1])
result = compute_metrics((dummy_logits, dummy_labels))
print('Test metrics:', result)


Test metrics: {'accuracy': 1.0, 'f1_macro': 1.0, 'balanced_accuracy': 1.0, 'precision_macro': 1.0, 'recall_macro': 1.0}


In [10]:
# CELL 11 ? Training configuration
#
# This cell is defensive against older/newer transformers versions by only
# passing kwargs that exist in the installed TrainingArguments signature.

import inspect

OUTPUT_DIR = './evidence_model_output'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

candidate_kwargs = {
    'output_dir': OUTPUT_DIR,
    'num_train_epochs': 3,
    'per_device_train_batch_size': 8,
    'per_device_eval_batch_size': 16,
    'gradient_accumulation_steps': 2,
    'learning_rate': 1e-5,
    'warmup_ratio': 0.1,
    'weight_decay': 0.02,
    'label_smoothing_factor': 0.05,
    'save_strategy': 'epoch',
    'logging_dir': f'{OUTPUT_DIR}/logs',
    'logging_steps': 25,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'f1_macro',
    'greater_is_better': True,
    'fp16': torch.cuda.is_available(),
    'max_grad_norm': 1.0,
    'seed': 42,
    'report_to': 'none',
}

params = inspect.signature(TrainingArguments.__init__).parameters
if 'evaluation_strategy' in params:
    candidate_kwargs['evaluation_strategy'] = 'epoch'
elif 'eval_strategy' in params:
    candidate_kwargs['eval_strategy'] = 'epoch'

supported_kwargs = {k: v for k, v in candidate_kwargs.items() if k in params}
unsupported_kwargs = sorted(k for k in candidate_kwargs if k not in params)

# Ensure best-model loading only stays on if the version supports the required knobs.
required_for_best = {'load_best_model_at_end', 'metric_for_best_model'}
if not required_for_best.issubset(supported_kwargs.keys()):
    supported_kwargs.pop('load_best_model_at_end', None)
    supported_kwargs.pop('metric_for_best_model', None)
    supported_kwargs.pop('greater_is_better', None)

training_args = TrainingArguments(**supported_kwargs)

actual_eval_strategy = getattr(training_args, 'evaluation_strategy', None)
if actual_eval_strategy is None:
    actual_eval_strategy = getattr(training_args, 'eval_strategy', None)

print('Training config:')
print(f'  Epochs:               {getattr(training_args, "num_train_epochs", "n/a")}')
print(f'  Batch size:           {getattr(training_args, "per_device_train_batch_size", "n/a")}')
print(f'  Grad accumulation:    {getattr(training_args, "gradient_accumulation_steps", "n/a")}')
print(f'  Learning rate:        {getattr(training_args, "learning_rate", "n/a")}')
print(f'  Label smoothing:      {getattr(training_args, "label_smoothing_factor", "n/a")}')
print(f'  Mixed precision:      {getattr(training_args, "fp16", "n/a")}')
print(f'  Evaluation strategy:  {actual_eval_strategy}')
print(f'  Save strategy:        {getattr(training_args, "save_strategy", "n/a")}')
print(f'  Output dir:           {OUTPUT_DIR}')
if unsupported_kwargs:
    print(f'  Skipped unsupported args: {unsupported_kwargs}')


Training config:
  Epochs:               3
  Batch size:           8
  Grad accumulation:    2
  Learning rate:        1e-05
  Label smoothing:      0.05
  Mixed precision:      True
  Evaluation strategy:  IntervalStrategy.EPOCH
  Save strategy:        SaveStrategy.EPOCH
  Output dir:           ./evidence_model_output


In [11]:
# CELL 12 — Train the model
#
# The Trainer class handles:
# - batching data
# - forward pass (compute predictions)
# - backward pass (compute gradients)
# - updating weights
# - evaluating on validation set
# - saving checkpoints

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)
        # Stops training if val metric doesn't improve for 3 evals
    ],
)

print("Starting training...")
print(f"Training on {len(train_dataset):,} examples")
print(f"Validating on {len(val_dataset):,} examples")
print("-" * 50)

# TRAIN
train_result = trainer.train()

print("\n✅ Training complete!")
print(f"Final train loss: {train_result.training_loss:.4f}")

Starting training...
Training on 15,000 examples
Validating on 1,998 examples
--------------------------------------------------


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Balanced Accuracy,Precision Macro,Recall Macro
1,0.388400,0.396631,0.891400,0.891000,0.891400,0.890800,0.891400
2,0.229100,0.429811,0.896400,0.896000,0.896400,0.896600,0.896400
3,0.227700,0.444654,0.894400,0.894200,0.894400,0.894300,0.894400



✅ Training complete!
Final train loss: 0.3371


In [12]:
# CELL 13 ? Full evaluation with classification report

print("Evaluating on validation set...")
predictions_output = trainer.predict(val_dataset)
logits = predictions_output.predictions
true_labels = predictions_output.label_ids

# Convert logits to probabilities using softmax
def softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

probs = softmax(logits)
pred_labels = np.argmax(probs, axis=1)

print("\nClassification Report:")
print(classification_report(
    true_labels,
    pred_labels,
    labels=[0, 1, 2],
    target_names=['support', 'contradict', 'neutral'],
    zero_division=0,
))

print("Confusion Matrix (rows=true, cols=predicted):")
cm = confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2])
cm_df = pd.DataFrame(
    cm,
    index=['true_support', 'true_contradict', 'true_neutral'],
    columns=['pred_support', 'pred_contradict', 'pred_neutral'],
)
print(cm_df)

sample_count = min(5, len(val_examples), len(true_labels), len(pred_labels), len(probs))
print(f"\n--- Sample predictions ({sample_count}) ---")
for i in range(sample_count):
    true = ID_TO_LABEL[int(true_labels[i])]
    pred = ID_TO_LABEL[int(pred_labels[i])]
    conf = float(probs[i][pred_labels[i]])
    marker = 'OK' if true == pred else 'X'
    ex = val_examples[i]
    print(f"{marker} Claim: {ex['claim'][:60]}...")
    print(f"   True: {true:12} | Predicted: {pred:12} | Confidence: {conf:.3f}")
    print()


Evaluating on validation set...



Classification Report:
              precision    recall  f1-score   support

     support       0.85      0.89      0.87       666
  contradict       0.89      0.83      0.86       666
     neutral       0.95      0.97      0.96       666

    accuracy                           0.90      1998
   macro avg       0.90      0.90      0.90      1998
weighted avg       0.90      0.90      0.90      1998

Confusion Matrix (rows=true, cols=predicted):
                 pred_support  pred_contradict  pred_neutral
true_support              592               57            17
true_contradict            92              554            20
true_neutral               12                9           645

--- Sample predictions (5) ---
OK Claim: Edmund H. North won an Emmy Award....
   True: neutral      | Predicted: neutral      | Confidence: 0.971

OK Claim: Stripes had a person appear in it....
   True: support      | Predicted: support      | Confidence: 0.962

OK Claim: Guatemala's civil war was fou

In [13]:
import json
import os

# CELL 14 ? Save model and tokenizer
#
# We save:
# 1. The model weights
# 2. The tokenizer
# 3. The Hugging Face config
# 4. Our custom label mappings and metadata

SAVE_PATH = './evidence_classifier_model'
Path(SAVE_PATH).mkdir(exist_ok=True)

print(f'Saving model to: {SAVE_PATH}')
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

label_config = {
    'id2label': {str(k): v for k, v in ID_TO_LABEL.items()},
    'label2id': {v: k for k, v in ID_TO_LABEL.items()},
    'label_map': LABEL_MAP,
    'model_name': MODEL_NAME,
    'max_length': MAX_LENGTH,
    'training_note': 'Weak-supervision baseline trained on FEVER claims paired with Wikipedia page titles / sampled hard negatives.',
}
with open(f'{SAVE_PATH}/label_config.json', 'w', encoding='utf-8') as f:
    json.dump(label_config, f, indent=2)

print('Saved artifacts:')
print(f'  Model:        {SAVE_PATH}/')
print(f'  Label config: {SAVE_PATH}/label_config.json')

files = os.listdir(SAVE_PATH)
for f in sorted(files):
    size = os.path.getsize(f'{SAVE_PATH}/{f}')
    print(f"  {f:<40} {size/1024/1024:.1f} MB" if size > 1024*1024 else f"  {f}")


Saving model to: ./evidence_classifier_model
Saved artifacts:
  Model:        ./evidence_classifier_model/
  Label config: ./evidence_classifier_model/label_config.json
  config.json
  label_config.json
  model.safetensors                        703.5 MB
  special_tokens_map.json
  tokenizer.json                           8.2 MB
  tokenizer_config.json
  training_args.bin


In [14]:
# CELL 15 ? Test inference on new claim+document pairs
#
# This is what your pipeline will use.
# Load model -> tokenize -> forward pass -> get probabilities

from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch, json

INFER_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def load_evidence_classifier(model_path='./evidence_classifier_model'):
    """Load saved model for inference."""
    if not Path(model_path).exists():
        raise FileNotFoundError(f'Model path not found: {model_path}. Run the training and save cells first.')

    print(f'Loading from: {model_path}')
    tok = AutoTokenizer.from_pretrained(model_path)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_path)
    mdl = mdl.to(INFER_DEVICE)
    mdl.eval()

    with open(f'{model_path}/label_config.json', encoding='utf-8') as f:
        cfg = json.load(f)

    return tok, mdl, cfg


def predict_evidence(claim, document_text, tokenizer, model, config, max_length=None):
    max_length = max_length or config.get('max_length', 320)
    inputs = tokenizer(
        claim,
        document_text[:1000],
        max_length=max_length,
        truncation=True,
        padding='max_length',
        return_tensors='pt',
    )
    inputs = {k: v.to(INFER_DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    probs = torch.softmax(logits, dim=1)[0].detach().cpu()
    pred_id = int(probs.argmax().item())
    id2label = config['id2label']
    label = id2label[str(pred_id)]
    confidence = float(probs[pred_id].item())
    probabilities = {
        id2label[str(i)]: round(float(probs[i].item()), 4)
        for i in range(len(probs))
    }

    return {
        'label': label,
        'confidence': round(confidence, 4),
        'probabilities': probabilities,
    }


tok, mdl, cfg = load_evidence_classifier('./evidence_classifier_model')

test_cases = [
    {
        'claim': 'The Eiffel Tower is located in Paris',
        'document': 'The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.',
        'expected': 'support',
    },
    {
        'claim': 'Coffee causes cancer',
        'document': 'Multiple studies have shown that moderate coffee consumption is not linked to cancer and may even reduce certain cancer risks.',
        'expected': 'contradict',
    },
    {
        'claim': 'Python was invented in 1991',
        'document': 'The history of computing includes many programming languages developed across different eras.',
        'expected': 'neutral',
    },
]

print('Testing inference on new examples:\n')
for tc in test_cases:
    result = predict_evidence(tc['claim'], tc['document'], tok, mdl, cfg)
    marker = 'OK' if result['label'] == tc['expected'] else 'X'
    print(f"{marker} Claim:      {tc['claim']}")
    print(f"  Document:    {tc['document'][:70]}...")
    print(f"  Expected:    {tc['expected']}")
    print(f"  Predicted:   {result['label']} (confidence: {result['confidence']:.3f})")
    print('  All probs:   ' + '  '.join(f"{k}={v:.3f}" for k, v in result['probabilities'].items()))
    print()


Loading from: ./evidence_classifier_model


The tokenizer you are loading from './evidence_classifier_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Testing inference on new examples:

OK Claim:      The Eiffel Tower is located in Paris
  Document:    The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars ...
  Expected:    support
  Predicted:   support (confidence: 0.973)
  All probs:   support=0.973  contradict=0.018  neutral=0.009

OK Claim:      Coffee causes cancer
  Document:    Multiple studies have shown that moderate coffee consumption is not li...
  Expected:    contradict
  Predicted:   contradict (confidence: 0.581)
  All probs:   support=0.022  contradict=0.581  neutral=0.397

X Claim:      Python was invented in 1991
  Document:    The history of computing includes many programming languages developed...
  Expected:    neutral
  Predicted:   support (confidence: 0.966)
  All probs:   support=0.966  contradict=0.028  neutral=0.007



In [15]:
# CELL 16 ? EvidenceClassifier wrapper class
#
# This is the class that goes into your pipeline code.
# It matches the interface your evidence.py expects more closely.

class EvidenceClassifier:
    """
    Drop-in ML replacement for the LLM-based evidence classifier.

    Usage:
        clf = EvidenceClassifier('./evidence_classifier_model')
        result = clf.classify(claim, document_text)
    """

    def __init__(self, model_path='./evidence_classifier_model'):
        self.model_path = model_path
        self.loaded = False
        self._try_load()

    def _try_load(self):
        try:
            self.tokenizer, self.model, self.config = load_evidence_classifier(self.model_path)
            self.loaded = True
            print(f'Loaded EvidenceClassifier from {self.model_path}')
        except Exception as e:
            print(f'Could not load ML model: {e}')
            print('Will use heuristic fallback')
            self.loaded = False

    def classify(self, claim: str, document_text: str) -> dict:
        if self.loaded:
            try:
                result = predict_evidence(claim, document_text, self.tokenizer, self.model, self.config)
                return {
                    'label': result['label'],
                    'confidence_score': result['confidence'],
                    'probabilities': result['probabilities'],
                    'analysis_method': 'model',
                    'reasoning': self._template_reasoning(result['label'], result['confidence']),
                    'evidence_excerpt': self._extract_excerpt(document_text),
                }
            except Exception as e:
                print(f'ML inference failed: {e}, using heuristic')

        return self._heuristic_classify(claim, document_text)

    def _template_reasoning(self, label, confidence):
        pct = int(confidence * 100)
        templates = {
            'support': f'ML classifier determined this document supports the claim ({pct}% confidence).',
            'contradict': f'ML classifier determined this document contradicts the claim ({pct}% confidence).',
            'neutral': f'ML classifier determined this document is neutral toward the claim ({pct}% confidence).',
        }
        return templates.get(label, 'Classification performed by ML model.')

    def _extract_excerpt(self, text, max_chars=200):
        sentences = text.replace('\n', ' ').split('.')
        for sent in sentences:
            sent = sent.strip()
            if len(sent) > 30:
                return sent[:max_chars] + ('...' if len(sent) > max_chars else '')
        return text[:max_chars]

    def _heuristic_classify(self, claim, document_text):
        claim_lower = claim.lower()
        doc_lower = document_text.lower()
        claim_words = set(claim_lower.split()) - {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'in', 'of', 'to', 'that'}
        matches = sum(1 for w in claim_words if w in doc_lower)
        overlap = matches / max(len(claim_words), 1)
        contradiction_signals = ['not', 'never', 'false', 'incorrect', 'wrong', 'disputed', 'contrary', 'refutes', 'debunked']
        has_contradiction = any(sig in doc_lower for sig in contradiction_signals)

        if overlap > 0.4 and has_contradiction:
            label, confidence = 'contradict', 0.55
        elif overlap > 0.5:
            label, confidence = 'support', 0.55
        elif overlap > 0.2:
            label, confidence = 'neutral', 0.6
        else:
            label, confidence = 'neutral', 0.5

        return {
            'label': label,
            'confidence_score': confidence,
            'probabilities': {'support': 0.33, 'contradict': 0.33, 'neutral': 0.34},
            'analysis_method': 'heuristic',
            'reasoning': f'Heuristic classification based on keyword overlap ({overlap:.2f}).',
            'evidence_excerpt': document_text[:200],
        }


print('Testing EvidenceClassifier wrapper:\n')
clf = EvidenceClassifier('./evidence_classifier_model')

test_claim = 'Vaccines are effective at preventing disease'
test_doc = 'Clinical trials have demonstrated that vaccines provide strong protection against infectious diseases, with efficacy rates typically above 90%.'

result = clf.classify(test_claim, test_doc)
print(f'Claim:    {test_claim}')
print(f'Document: {test_doc[:70]}...')
print('\nResult:')
for k, v in result.items():
    print(f'  {k}: {v}')


Testing EvidenceClassifier wrapper:

Loading from: ./evidence_classifier_model


The tokenizer you are loading from './evidence_classifier_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loaded EvidenceClassifier from ./evidence_classifier_model
Claim:    Vaccines are effective at preventing disease
Document: Clinical trials have demonstrated that vaccines provide strong protect...

Result:
  label: support
  confidence_score: 0.9721
  probabilities: {'support': 0.9721, 'contradict': 0.0223, 'neutral': 0.0056}
  analysis_method: model
  reasoning: ML classifier determined this document supports the claim (97% confidence).
  evidence_excerpt: Clinical trials have demonstrated that vaccines provide strong protection against infectious diseases, with efficacy rates typically above 90%
